# GAN variants (DCGAN, WGAN, conditional) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## stabilizing and steering GAN training
The vanilla GAN game from 10.5 is the base. Convolutions add image-like inductive bias, Wasserstein distance changes the critic signal, and conditioning injects labels or attributes. These modifications set up StyleGAN and CycleGAN (10.7) and make evaluation questions in 10.8 sharper.

In [ ]:
# 1) Wasserstein distance between shifted point clouds tracks mean gap
real=np.array([0.,1.,2.]); fake=np.array([.5,1.5,2.5]); gap=fake.mean()-real.mean()
plt.figure(figsize=(4,3)); plt.bar(['real mean','fake mean'],[real.mean(),fake.mean()]); plt.title('transport gap between means')
assert abs(gap-0.5)<1e-12

In [ ]:
# 2) clipping constrains a critic slope
x=np.linspace(-2,2,100); slopes=[0.5,1.0,2.0]
plt.figure(figsize=(5,3))
for s in slopes: plt.plot(x,np.clip(s*x,-1,1),label=f's={s}')
plt.legend(); plt.title('critic constrained by Lipschitz-like clipping')
assert np.max(np.abs(np.clip(2*x,-1,1)))<=1

In [ ]:
# 3) conditional GAN separates two target modes by label
z=np.linspace(-1,1,50); y0=z-1; y1=z+1
plt.figure(figsize=(5,3)); plt.plot(z,y0,label='label 0'); plt.plot(z,y1,label='label 1'); plt.legend(); plt.title('same noise, different condition')
assert np.allclose(y1-y0,2)

In [ ]:
# 4) DCGAN intuition: convolutional smoothing preserves locality
signal=np.r_[np.zeros(20),np.ones(20),np.zeros(20)]; kernel=np.array([.25,.5,.25]); smooth=np.convolve(signal,kernel,mode='same')
plt.figure(figsize=(5,3)); plt.plot(signal,label='input'); plt.plot(smooth,label='conv'); plt.legend(); plt.title('local convolution shares structure')
assert smooth[30]==1.0

In [ ]:
# 5) variant choice changes the training signal near saturation
Df=np.linspace(.01,.99,100); vanilla=-np.log(Df); wasser=np.abs(Df-.5)
plt.figure(figsize=(5,3)); plt.plot(Df,vanilla,label='non-saturating'); plt.plot(Df,wasser,label='toy W gap'); plt.legend(); plt.title('different objectives shape gradients')
assert vanilla[0]>vanilla[-1]